In [1]:
from datasets import load_dataset

ds = load_dataset("malaysia-ai/Emilia-YODAS-Voice-Conversion", "audio_text")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
Generating train split: 100%|██████████| 11365354/11365354 [00:02<00:00, 4741210.76 examples/s]


In [11]:
from multiprocess import Pool
import itertools

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [12]:
from tqdm import tqdm

def loop(indices):
    indices, _ = indices
    ds = load_dataset("malaysia-ai/Emilia-YODAS-Voice-Conversion", "audio_text")
    data = set()
    for i in tqdm(indices):
        data.add(ds['train'][i]['audio_filename'])
    return [data]

In [14]:
data = multiprocessing(range(len(ds['train'])), loop, cores = 20)

100%|██████████| 14/14 [00:00<00:00, 25777.11it/s]


In [15]:
len(data)

21

In [16]:
combined = set()
for d in data:
    combined.update(d)

In [19]:
import json

with open('Emilia-YODAS-Voice-Conversion-audio.json', 'w') as fopen:
    json.dump(list(combined), fopen)